# 05. Transferability and boundary thickness (RQ3)

This notebook makes the transfer figures and the boundary-thickness results:

- **Figure 1** (`fig_transfer_constrained`): transfer rates of white-box evasions between every pair of models, constrained attacker.
- **Figure S3** (`fig_transfer_unconstrained`): the same, unconstrained attacker.
- **Table S-XXIV** (`reports/table4_thickness_corr.csv`): does boundary thickness predict robustness? Within-dataset rank correlations, also the RQ3 numbers of the text.
- **Figure S5** (`fig_thickness_vs_asr`): thickness against worst-case ASR.

**Inputs:** `runs/<dataset>/attacks/transfer_{constrained,unconstrained}.npz` (`fraudadv.attacks.transfer`, from the grid's `adv_<mode>_<model>.npz`), the grid's thickness along adversarial directions (`thickness*.csv`), the class-pair thickness (`thickness_pairs.csv`, `fraudadv.boundary.run_thickness`) and the grid's `results*.csv`. The live cell also reads the saved Sparkov MLP and XGBoost, `runs/sparkov/attacks/tau.json`, `adv_constrained_{mlp,xgboost}.npz` and `data/processed/sparkov`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

## Figure 1 and Figure S3: transfer rates

`make_report.fig_transfer` reads each dataset's transfer matrix. A cell is the share of a source model's high-confidence white-box evasions ($\epsilon = 0.5$) that also evade the target model, among the frauds both detect. The figure has one heatmap per dataset and a panel pooled over datasets, weighted by the number of replayed evasions. It is drawn for the constrained attacker (Figure 1) and the unconstrained one (Figure S3).

In [ ]:
(ROOT / "reports").mkdir(exist_ok=True)

from fraudadv.report import make_report as MR

for mode in ("constrained", "unconstrained"):
    MR.fig_transfer(mode)           # reports/fig_transfer_<mode>.{png,pdf}
nb.show_figure("fig_transfer_constrained")

In [ ]:
nb.show_figure("fig_transfer_unconstrained")

## Run it yourself: one transfer rate, recomputed

This cell recomputes one cell of Figure 1 from the stored adversarial examples. On Sparkov, it takes the constrained MLP evasions and replays them on the XGBoost model at its 1%-FPR threshold, with the library's own function (`fraudadv.attacks.transfer.matrix`, restricted to these two models). It then prints the rate beside the stored value in `runs/sparkov/attacks/transfer_constrained.npz`. The function loads both models and the stored examples, takes a few seconds and writes nothing. The rate is one of the study's cells (the Sparkov panel of Figure 1), not a sample.

In [ ]:
import json

import numpy as np

from fraudadv.attacks import transfer

DATASET, SOURCE, TARGET = "sparkov", "mlp", "xgboost"
attack_dir = ROOT / "runs" / DATASET / "attacks"
tau = json.loads((attack_dir / "tau.json").read_text())                 # the grid's 1%-FPR thresholds
T, N, _, _ = transfer.matrix(DATASET, "constrained", [SOURCE, TARGET], tau)
stored = np.load(attack_dir / "transfer_constrained.npz")
families = list(stored["families"])
s, t = families.index(SOURCE), families.index(TARGET)
print(f"{DATASET}, constrained, {SOURCE} -> {TARGET}: recomputed {T[0, 1]:.4f} on {int(N[0, 1])} evasions; "
      f"stored {stored['T'][s, t]:.4f} on {int(stored['N'][s, t])} evasions")

## Table S-XXIV: does boundary thickness predict robustness?

`make_report.thickness_analysis` joins each model's boundary thickness with its worst-case ASR. Two variants are used: thickness along each fraud's own adversarial direction (from the grid) and class-pair thickness (each detected fraud paired with a legitimate transaction the model passes). For each variant, threat model and budget it computes the mean within-dataset Spearman rank correlation across models, with a permutation $p$-value (models shuffled within datasets, 2,000 permutations; this takes about 45 seconds). It writes `reports/table4_thickness_corr.csv` (the RQ3 numbers of the text), `reports/table4_thickness.csv` (every cell) and Figure S5.

In [ ]:
A = MR.load_attacks()           # the grid: runs/<dataset>/attacks/results*.csv
MR.thickness_analysis(A)        # reports/table4_thickness_corr.csv, table4_thickness.csv, fig_thickness_vs_asr.{png,pdf}
nb.show_csv("reports/table4_thickness_corr.csv");

In [ ]:
nb.show_csv("reports/table4_thickness_corr.csv");

## Figure S5: thickness against worst-case ASR

The class-pair thickness of each model against its worst-case ASR at $\epsilon = 0.5$, one panel per threat model, titled with the within-dataset rank correlation (drawn by `thickness_analysis` above).

In [ ]:
nb.show_figure("fig_thickness_vs_asr")

## Rerun the runs behind these results from scratch

`scripts/reproduce.sh` makes the transfer matrices with `fraudadv.attacks.transfer` (both threat models). It replays the stored adversarial examples of the attack grid on every model, so the grid must exist first (notebook 03). The grid also writes the thickness along adversarial directions (`thickness*.csv`). The class-pair thickness (`thickness_pairs.csv`) comes from `fraudadv.boundary.run_thickness`. `scripts/reproduce.sh` does not call that step, so its call below follows `RESULTS.md`. These runs take from minutes (transfer) to hours (class-pair thickness on the large datasets). They write into `runs/<dataset>/attacks/` and replace the saved results, so they are off here (`RERUN = False`). Afterwards, run this notebook again from the top.

In [ ]:
RERUN = False
DATASET = "ulb"            # ulb | ieee_cis | sparkov | tabformer | paysim | baf

if RERUN:
    from fraudadv.attacks import transfer
    from fraudadv.boundary import run_thickness

    transfer.run(DATASET)                     # transfer_unconstrained.npz, transfer_constrained.npz (scripts/reproduce.sh)
    FAMILIES = ["xgboost", "catboost", "mlp", "ft_transformer", "ivis"]
    if DATASET not in ("ulb", "baf"):         # graph and foundation models: card/account sequences only
        FAMILIES += ["hgnn", "hgt", "chronos2", "timesfm"]
    run_thickness.run(DATASET, FAMILIES)      # thickness_pairs.csv (not in scripts/reproduce.sh; RESULTS.md)

## Check against the reference

Every file this notebook wrote under `reports/`, compared byte for byte with the reference build.

In [ ]:
nb.check([f"reports/fig_transfer_{mode}.{ext}" for mode in ("constrained", "unconstrained") for ext in ("pdf", "png")]
         + ["reports/table4_thickness_corr.csv", "reports/table4_thickness.csv"]
         + [f"reports/fig_thickness_vs_asr.{ext}" for ext in ("pdf", "png")]);